## DAPO 的三板斧（仍用 GRPO 目标，只换“ Trick 包”）

- Clip-Higher（非对称裁剪）

<b>把“上限”从 1+ε 拉到 1+ε_high（默认 0.28），下限保持 1−ε（0.2）→ 低概率好 token 可以大幅拉升，熵不掉 0 。</b>

- Dynamic Sampling（动态重采）

若一批样本全 1/全 0，立即丢弃并重新生成，直到组内同时存在“对 & 错”回答，保证 advantage≠0 。

- Token-Level Loss（token 平均代替样本平均）

先把每条样本内所有 token 的梯度求和，再在 batch 维度平均 → 长序列不再被“样本级归一”稀释，高质量长 CoT 也能贡献足够梯度 。

## 1 安装&数据


In [ ]:
pip install transformers datasets torch accelerate

# 在命令行执行以下命令
# 生成 1k 条小学应用题
python -c "
from datasets import load_dataset
ds = load_dataset('gsm8k','main')['train'].select(range(1000))
ds.to_json('gsm1k.jsonl')
"

In [ ]:
# mini-dapo.yaml 文件中
model_name: microsoft/DialoGPT-medium   # 小模型就能跑
max_new_tokens: 256
clip_low: 0.2
clip_high: 0.28        # Clip-Higher
dynamic_sample: true   # 简单版：组内全同 acc 就重采
train_batch: 64
gen_batch: 128
lr: 5e-6
epochs: 1
kl_coef: 0.1

## 训练脚本
- DAPO-mini: 序列级 reward + Clip-Higher + 动态采样 + token-mean loss
- 运行：python dapo_mini.py


In [ ]:
import torch,json,random,math,yaml
from transformers import AutoTokenizer,AutoModelForCausalLM
from datasets import load_dataset

cfg = yaml.safe_load(open('mini-dapo.yaml'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(cfg['model_name'])
tok_pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(cfg['model_name']).to(device)

ref_model = AutoModelForCausalLM.from_pretrained(cfg['model_name']).to(device)
optimizer = torch.optim.AdamW(model.parameters(),lr=cfg['lr'])

def extract_answer(text):
    """提取：最后一个数字（即答案）"""
    # 把字符串 text 里所有“纯数字单词”提取出来，转成 int 列表
    nums = [int(s) for s in text.replace(',','').split() if s.isdigit()]
    return num[-1] if nums else -1

def reward_fn(question,completion):
    """1 ：对，0：错"""
    gold = extract_answer(question)
    pred = extract_answer(completion)
    return 1.0 if pred == gold else 0.0
@torch.no_grad()
def generate_batch(questions):
    """返回 （text,log_prob）列表"""
    prompts = [f"Q:{q}\nA:" for q in questions]
    inputs = tok(prompts,return_tensors='pt',padding=True).to(device)
    print(inputs.keys())
    print(inputs.input_ids.shape)
    print(inputs.attention_mask)
    max_new = cfg['max_new_tokens']
    outputs = model.generate(
        **inputs,max_new_tokens=max_new,do_sample=True,temperature=0.7,
        return_dict_in_generate=True,output_scores=True
    )
    # 计算每条序列 log π
    scores = torch.stack(outputs.scores,dim=0) # [L,B,V]
    logp_seq =[]
    
    for b in range(scores.shape[1]):
        lp = 0.0
        for t in range(scores.shape[0]):
            probs = scores[t,b].log_softmax(-1)
            # 第 b 条样本、第 t 个新生成 token 的 id
            tok_id = outputs.sequences[b,inputs.input_ids.shape[1] + t]
            # 当前句子所有token的probs之和,得到整条回答的似然
            lp +=probs[tok_id].item()
        logp_seq.append(lp)
    # 将token id解码为可读文本
    texts = [tok.decode(s,skip_special_tokens=True) for s in outputs.sequences]
    return texts,logp_seq

# ------ 动态采样 ------
def dynamic_sample(groups):
    """当组内acc全为0或全为1时就从新采样"""
    while True:
        random.shuffle(groups)
        accs = [g['acc'] for g in groups[:cfg['train_batch']]]
        if len(set(acss)) >1:
            return groups[:cfg['train_batch']]
        # 否则重新采样

In [ ]:
# ------ 训练 ------
def train_step(batch):
    questions = [b['question'] for b in batch]
    texts_old,logp_old = generate_batch(question)
    rewards = [reward_fn(q,txt) for q,txt in zip(questions,texts_old)]
    
    # 组内归一化 advantage
    rewards = torch.tensor(rewards)
    adv = (rewards - rewards.mean()) / (rewards.std()+1e-8)
    
    # --- 重新做forward 获得新的logp + KL ---
    logp_new,kl = [],[]
    for txt in texts_old:
        inputs = tok(txt,return_tensors='pt').to(device)
        with torch.no_grad():
            logits_ref = ref_model(**inputs).logits[:,:-1,:]
        logits = model(**inputs).logits[:,:-1,:]
        logp = -torch.nn.functional.cross_entropy(
            logits.view(-1,logits.size(-1)),
            inputs.input_ids[:,1:].view(-1),
            reduction ='sum'
        ).item()
        logp_new.append(logp)
        # KL = (log π_ref  - log π)
        # 得到整条回答的近似 KL 值
        # gather 取出实际采样到的那个 token 的 log-ratio，形状 [B, L, 1]
        # .sum().item()
        # 把 batch 内、长度维度全部累加，得到标量 KL
        kl.append(
            (logits_ref.log_softmax(-1) - logits.log_softmax(-1)).gather(2,inputs.input_ids[:,1:].unsqueeze(2)).sum().item()
        )
    logp_old = torch.tensor(logp_old)
    logp_new = torch.tensor(logp_new)
    
    ratio = torch.exp(logp_new - logp_old)
    kl = torch.tensor(kl)
    
    # ------ Clip-Higher ------
    clip_low = 1 - cfg['clip_low']
    clip_high = 1 + cfg['clip_high']
    
    surr1 = ratio * adv
    surr2 = torch.clamp(ratio,clip_low,clip_high) * adv
    pg_loss = -torch.min(surr1,surr2).mean()
    kl_loss = cfg['kl_coef'] * kl.mean()
    loss = pg_loss + kl_loss
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return rewards.mean().item()

# ------ 主循环 ------
ds = load_dataset('json', data_files='gsm1k.jsonl')['train']
for epoch in range(cfg['epochs']):
    groups = [{'question': ex['question'], 'answer': ex['answer']} for ex in ds]
    random.shuffle(groups)
    pbar = range(0, len(groups), cfg['gen_batch'])
    for i in pbar:
        batch = dynamic_sample(groups[i:i+cfg['gen_batch']])
        acc = train_step(batch)
        print(f"step {i//cfg['gen_batch']}  acc={acc:.3f}")
    # 保存
    model.save_pretrained(f"dapo-epoch{epoch}")
    print(">>> epoch", epoch, "done <<<")